# 1. Imports

In [170]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [171]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [172]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("season_database").getOrCreate()

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [173]:
# events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

# season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# # Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
# df_events = spark.read.parquet(*season_events_parquet_file_paths)

# df_events = df_events.withColumnsRenamed({
#     "id": "eventId",
#     "player.id": "eventPlayer.id",
#     "player.name": "eventPlayer.name",
#     "team.id": "eventTeam.id",
#     "team.name": "eventTeam.name",
# })

# df_events.select('balls').show(truncate=False)

In [174]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("z", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    'competitionId',
    'season', # dps mudar pra seasonId se necessário
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    'details_parsed',
    #'homeTeam',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [175]:
print('Quantidade de linhas:', df_events.count())

Quantidade de linhas: 945154


## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [176]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw.withColumns({
    "homeTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.id`')).otherwise(F.col('`opponentTeam.id`')),
    "homeTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`team.name`')).otherwise(F.col('`opponentTeam.name`')),

    "opponentTeamId": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.id`')).otherwise(F.col('`opponentTeam.id`')),
    "opponentTeamName": F.when(F.col('venueType') == 'TEAM_HOME', F.col('`opponentTeam.name`')).otherwise(F.col('`opponentTeam.name`')),
    }).select(
        'gameId', 
        'date',
        'season',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'),
        'homeTeamId',
        'homeTeamName',
        'opponentTeamId',
        'opponentTeamName',
        F.col('teamExtraTimeStartSide').alias('homeTeamExtraTimeStartSide'), 
        F.col('teamStartSide').alias('homeTeamStartSide'),
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

df_games.show(5)

+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+--------------------------+-----------------+-------------+-------------+------------+
|gameId|      date|   season|competitionId|competitionName|homeTeamId|        homeTeamName|opponentTeamId|    opponentTeamName|homeTeamExtraTimeStartSide|homeTeamStartSide|  stadiumName|stadiumLength|stadiumWidth|
+------+----------+---------+-------------+---------------+----------+--------------------+--------------+--------------------+--------------------------+-----------------+-------------+-------------+------------+
|  4447|2022-08-13|2022-2023|            1| Premier League|         3|         Aston Villa|             8|             Everton|                     Right|             Left|   Villa Park|        105.0|        68.0|
|  4760|2023-04-25|2022-2023|            1| Premier League|        20|Wolverhampton Wan...|            20|Wolverhampton Wan...|                 

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [177]:
df_games_events = df_events.join(df_games.drop('season', 'competitionId', 'competitionName'), on = "gameId", how='left')

df_games_events = (
    df_games_events.withColumn(
        'homeTeamAttackingDirection',
            F.when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Right')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Left')), 
                'Left'
            )
            .when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Left')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Right')), 
                'Right'
            )
        ).withColumn(
            'opponentTeamAttackingDirection',
            F.when(F.col('homeTeamAttackingDirection') == 'Right', 'Left')
            .when(F.col('homeTeamAttackingDirection') == 'Left', 'Right')
        )
)

df_games_events.show(5)

+------+-------------+---------+--------------------+------------+--------------------+------+-----------------+-----------------------+--------------+--------------------+-------------+---------------+-----------+-------------+--------------------+--------------------+--------------------+----------+----------+---------------+--------------+----------------+--------------------------+-----------------+----------------+-------------+------------+--------------------------+------------------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|periodDescription|startFormattedGameClock|startGameClock|      details_parsed|eventPlayerId|eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|homeTeamExtraTimeStartSide|homeTeamStartSide|     stadiumName|stadiumLength|stadiumWidth|homeTeamAttackingDirection|opponentTeamAt

## Domínios

In [178]:
# tabelas:
# partida - match_id
# competition -> competition_id
# season -> season_id
# events -> event_id

# df_defensive_events (domínio de desempenho técnico defensivo):
# competition_id | season_id | match_id | event_id | period | match_timestamp | event_type | event_name | atk_team | def_team | match_time_atk | match_time_def

# cruzamento dos dfs (pelo competitionID-seasonID-matchID) -> apenas deixar preparado para qnd fosse expandir, mas vamos usar o mesmo competitionID-seasonID:

# cria df_defensive_events -> cria df_threat_events -> outer join ou union dos dois pelo competitionID-seasonID-matchID (apenas deixar preparado para qnd fosse expandir, mas vamos usar o mesmo competitionID-seasonID) -> calcular o diff_threat_score

### Tipos de eventos:

- FIRSTKICKOFF: Inicio do primeiro tempo
- SECONDKICKOFF: Inicio do segundo tempo
- TC: Touch
- RE: Rebound
- BC: Ball Carry
- CL: Clearance
- CR: Cross
- CH: Challenge 
- OTB: A possession with a player on the ball
- PA: Pass
- FO: Foul
- FOUL: Additional foul
- SH: Shot

### Domínio: Desempenho Técnico Defensivo

### Eventos que queremos (Defensivos):

- Ofensivos como CR, PA e SH queremos que o resultado dele seja uma interferência da defesa adversária
- Defensivos como CL, CH, FO queremos que o tipo seja ação defensiva

- CL: Clearance
    - Qualquer CLEARANCE_OUTCOME_TYPE (A,B,D,E,O,P,S,U)
    - obs: talvez não E e U pq são FairPlay
    - obs2: P - Player e S - Stoppage não sei oq sejam, mas vou deixar

- CR: Cross
    - CROSS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- CH: Challenge. 
    - CHALLENGE_TYPE:
    - ‘5’ - 50/50. This is a duel type where two players compete for a loose ball.
    - A - Aerial duel. As the name suggests a duel type similar to 50-50, but with the ball coming from above.
    - B - Tackle from behind. As the name suggests a tackle attempt where the carrier puts their body in between the ball and the challenger as the tackle is attempted.
    - D - Dribble. The player tries to take on a defender in an attempt to get past them.
    - G - Goalkeeper smothers ball. A duel between the goalkeeper and a line player where the ball is loose and the goalkeeper tries to capture the ball.
    - H - Shielding. Similar to tackle from behind, but on a shielding challenge the carrier actively shields a defender who does not attempt a tackle
    - K - Hand tackle by goalkeeper. Despite the name, it is a duel type similar to goalkeeper smothers, but the keeper tries to parry the ball rather than retain it.
    - L - Slide tackle. Tackle type where the challenger slides to attempt to win the ball. Note that a player could be sliding on a dribble or 50-50, to be classed as a slide tackle it needs to be first and foremost a tackle.
    - S - Shoulder to shoulder. Tackle type where the challenger tries to win the ball with physical contact initiated with the body.
    - T - Standing tackle. Tackle attempt, usually from the front or side, that does not fit the other tackle types 
    - OBS1: **Único que não entraria como AD aqui seria o 'D'.**
    - OBS2: **Não estamos considerando outcome dos eventos.**

- PA: Pass
    - PASS_OUTCOME_TYPE:
    - B - Blocked
    - D - Defensive Interception

- FO: Foul
    - qualquer FOUL_TYPE = A, I, M
    - não vi evento de penalti, então teria q pegar a região dentro da area e evento de falta marcado ali (FOUL_TYPE == I)

- FOUL: Additional foul
    - são faltas adicionais no mesmo lance divida em mais de um evento, mas nos dados fica tudo NULL, então n vou add. FO já tem o evento principal de falta

- SH: Shot
    - SHOT_OUTCOME_TYPE:
    - B - Block on target. (Ball was going on target, but got blocked)
    - C - Block off target. (Ball was going off target, and got blocked)
    - F - Save off target. (Ball was going off target when it got saved)
    - L - Goalline clearance. (Ball is past the goalkeeper and a defender stops it from going into the net)
    - S - Save on target. (Ball was going on target and got saved).

In [179]:
desired_events = ['CL', 'CR', 'CH', 'PA', 'FO', 'SH'] # não inclui FO pq tá estranho misturando eventos dos outros e não de falta em si (dps validar isso)

cl_cond = (
    (F.col("eventType") == "CL") &
    F.col("details_parsed")["clearanceOutcomeType"].isin(['A','B','D','O','P','S'])
)

cr_cond = (
    (F.col("eventType") == "CR") &
    F.col("details_parsed")["crossOutcomeType"].isin(['B','D'])
)

ch_cond = (
    (F.col("eventType") == "CH") &
    ~F.col("details_parsed")["challengeType"].isin(['D'])
)

pa_cond = (
    (F.col("eventType") == "PA") &
    F.col("details_parsed")["passOutcomeType"].isin(['B','D'])
)

sh_cond = (
    (F.col("eventType") == "SH") &
    F.col("details_parsed")["shotOutcomeType"].isin(['B','C','F','L','S'])
)

In [180]:
df_defensive_events = (
    df_games_events
    .filter(F.col("eventType").isin(desired_events))
    .withColumns({
        "defensiveType": (
            F.when(cl_cond, F.col("details_parsed")["clearanceOutcomeType"])
            .when(cr_cond, F.col("details_parsed")["crossOutcomeType"])
            .when(ch_cond, F.col("details_parsed")["challengeType"])
            .when(pa_cond, F.col("details_parsed")["passOutcomeType"])
            .when(sh_cond, F.col("details_parsed")["shotOutcomeType"])
        ),

        "defensiveDescription": (
            F.when(cl_cond, F.col("details_parsed")["clearanceOutcomeTypeDescription"])
            .when(cr_cond, F.col("details_parsed")["crossOutcomeTypeDescription"])
            .when(ch_cond, F.col("details_parsed")["challengeTypeDescription"])
            .when(pa_cond, F.col("details_parsed")["passOutcomeTypeDescription"])
            .when(sh_cond, F.col("details_parsed")["shotOutcomeTypeDescription"])
        ),
        "eventTeamType": (
            F.when(F.col("eventType").isin(['CL', 'CH']), F.lit('Defending'))
            .when(F.col("eventType").isin(['CR', 'PA', 'SH']), F.lit('Attacking'))
        )
    }).dropna(subset=['defensiveType'])
)

df_defensive_events.show()

+------+-------------+---------+--------------------+---------+--------------------+------+-----------------+-----------------------+--------------+--------------------+-------------+-----------------+-----------+---------------+--------------------+--------------------+--------------------+----------+----------+---------------+--------------+----------------+--------------------------+-----------------+----------------+-------------+------------+--------------------------+------------------------------+-------------+--------------------+-------------+
|gameId|competitionId|   season|             eventId|eventType|eventTypeDescription|period|periodDescription|startFormattedGameClock|startGameClock|      details_parsed|eventPlayerId|  eventPlayerName|eventTeamId|  eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|homeTeamExtraTimeStartSide|homeTeamStartSide|     stadiumName|stadiumLength|sta

### Domínio: Ameaça

Vamos criar as variáveis de ameaça para os dois times, respeitando os sentidos de ataques deles. Isso apenas para os eventos defensivos.

In [182]:
# eixo x = de uma trave a outra dividido pelo meio campo (esquerda = negativo, direita = positivo) -> [-52.5, +52.5]
# eixo y = de uma lateral (acima do meio campo = positivo, abaixo do meio campo = negativo) -> [-34,+34]

ball_x = F.get("balls_parsed", 0)["x"]
ball_y = F.get("balls_parsed", 0)["y"]

df_events_ball_corners_dist = (
    df_defensive_events
    .withColumns({
        # meio do campo até o gol da esquerda no plano cartesiano
        'stadiumLength_left': -F.col('stadiumLength') / 2,
        # meio do campo até o gol da direita no plano cartesiano
        'stadiumLength_right': F.col('stadiumLength') / 2,

        # meio do campo até a lateral superior no plano cartesiano
        'stadiumWidth_top': F.col('stadiumWidth') / 2,
        # meio do campo até a lateral inferior no plano cartesiano
        'stadiumWidth_bottom': -F.col('stadiumWidth') / 2,
    })
    .withColumns({
        # distância euclidiana do escanteio superior esquerdo até a bola
        'ball_distance_from_top_left_corner': F.sqrt(
            F.pow(ball_x - F.col('stadiumLength_left'), 2) +
            F.pow(ball_y - F.col('stadiumWidth_top'), 2)
        ),

        # distância euclidiana do escanteio inferior esquerdo até a bola
        'ball_distance_from_bottom_left_corner': F.sqrt(
            F.pow(ball_x - F.col('stadiumLength_left'), 2) +
            F.pow(ball_y - F.col('stadiumWidth_bottom'), 2)
        ),

        # distância euclidiana escanteio superior direito até a bola
        'ball_distance_from_top_right_corner': F.sqrt(
            F.pow(ball_x - F.col('stadiumLength_right'), 2) +
            F.pow(ball_y - F.col('stadiumWidth_top'), 2)
        ),

        # distância euclidiana do escanteio inferior direito até a bola
        'ball_distance_from_bottom_right_corner': F.sqrt(
            F.pow(ball_x - F.col('stadiumLength_right'), 2) +
            F.pow(ball_y - F.col('stadiumWidth_bottom'), 2)
        ),
    }).withColumns({
        'min_ball_distance_from_left_corners': F.least(F.col('ball_distance_from_top_left_corner'), F.col('ball_distance_from_bottom_left_corner')),
        'min_ball_distance_from_right_corners': F.least(F.col('ball_distance_from_top_right_corner'), F.col('ball_distance_from_bottom_right_corner'))
    }).withColumns({
        'home_team_attacking_dir_min_ball_dist': (
            F.when(F.col('homeTeamAttackingDirection') == 'Left', F.col('min_ball_distance_from_right_corners'))
            .when(F.col('homeTeamAttackingDirection') == 'Right', F.col('min_ball_distance_from_left_corners'))
        ),
        'opponent_team_attacking_dir_min_ball_dist': (
            F.when(F.col('opponentTeamAttackingDirection') == 'Left', F.col('min_ball_distance_from_right_corners'))
            .when(F.col('opponentTeamAttackingDirection') == 'Right', F.col('min_ball_distance_from_left_corners'))
        )
    }).drop('ball_distance_from_top_left_corner', 'ball_distance_from_bottom_left_corner', 'ball_distance_from_top_right_corner', 'ball_distance_from_bottom_right_corner', 'min_ball_distance_from_left_corners', 'min_ball_distance_from_right_corners')
)

#df_events_ball_corners_dist.show()

In [183]:
df_events_qtd_players = (
    df_events_ball_corners_dist
    .withColumns({
        # Quantidade de jogadores do time mandante entre o gol esquerdo e a bola
        'homePlayers_between_ball_left_goal': (
            F.size(
                F.filter(
                    F.col('homePlayers_parsed'),
                    lambda p: (
                        (p['x'] >= F.col('stadiumLength_left')) &
                        (p['x'] <= F.get('balls_parsed', 0)['x'])
                    )
                )
            )
        ),

        # Quantidade de jogadores do time visitante entre o gol esquerdo e a bola
        'awayPlayers_between_ball_left_goal': (
            F.size(
                F.filter(
                    F.col('awayPlayers_parsed'),
                    lambda p: (
                        (p['x'] >= F.col('stadiumLength_left')) &
                        (p['x'] <= F.get('balls_parsed', 0)['x'])
                    )
                )
            )
        ),

        # Quantidade de jogadores do time mandante entre a bola e o gol direito
        'homePlayers_between_ball_right_goal': (
            F.size(
                F.filter(
                    F.col('homePlayers_parsed'),
                    lambda p: (
                        (p['x'] <= F.col('stadiumLength_right')) &
                        (p['x'] >= F.get('balls_parsed', 0)['x'])
                    )
                )
            )
        ),

        # Quantidade de jogadores do time visitante entre a bola e o gol direito
        'awayPlayers_between_ball_right_goal': (
            F.size(
                F.filter(
                    F.col('awayPlayers_parsed'),
                    lambda p: (
                        (p['x'] <= F.col('stadiumLength_right')) &
                        (p['x'] >= F.get('balls_parsed', 0)['x'])
                    )
                )
            )
        )
    })

    # Soma dos dois times do lado esquerdo
    .withColumn(
        'players_between_ball_left_goal',
        F.col('homePlayers_between_ball_left_goal') +
        F.col('awayPlayers_between_ball_left_goal')
    )

    # Soma dos dois times do lado direito
    .withColumn(
        'players_between_ball_right_goal',
        F.col('homePlayers_between_ball_right_goal') +
        F.col('awayPlayers_between_ball_right_goal')
    )
).withColumns({
    # Time com superioridade numérica entre a bola e o gol esquerdo
    'numerical_superiority_team_left': (
        F.when(F.col('homePlayers_between_ball_left_goal') > F.col('awayPlayers_between_ball_left_goal'), 'Home Team')
        .when(F.col('homePlayers_between_ball_left_goal') < F.col('awayPlayers_between_ball_left_goal'), 'Opponent Team')
        .otherwise(F.lit('Equal'))
    ),
    # Time com superioridade numérica entre a bola e o gol direito
    'numerical_superiority_team_right': (
        F.when(F.col('homePlayers_between_ball_right_goal') > F.col('awayPlayers_between_ball_right_goal'), 'Home Team')
        .when(F.col('homePlayers_between_ball_right_goal') < F.col('awayPlayers_between_ball_right_goal'), 'Opponent Team')
        .otherwise(F.lit('Equal'))
    )
})

# Dataframe com as colunas separadas considerando o sentido do ataque
df_events_qtd_players = (
    df_events_qtd_players.withColumns({
        # Qtd total de jogadores entre a bola e o gol quando é o time mandante atacando
        'home_team_attacking_dir_players_between_ball_goal': (
                F.when(F.col('homeTeamAttackingDirection') == 'Left', F.col('players_between_ball_left_goal'))
                .when(F.col('homeTeamAttackingDirection') == 'Right', F.col('players_between_ball_right_goal'))
        ),
        # Qtd total de jogadores entre a bola e o gol quando é o time adversário atacando
        'opponent_team_attacking_dir_players_between_ball_goal': (
                F.when(F.col('opponentTeamAttackingDirection') == 'Left', F.col('players_between_ball_left_goal'))
                .when(F.col('opponentTeamAttackingDirection') == 'Right', F.col('players_between_ball_right_goal'))
        ),
        # Time com superioridade numérica entre a bola e o gol quando é o time mandante atacando
        'home_team_attacking_dir_numerical_superiority_team': (
                F.when(F.col('homeTeamAttackingDirection') == 'Left', F.col('numerical_superiority_team_left'))
                .when(F.col('homeTeamAttackingDirection') == 'Right', F.col('numerical_superiority_team_right'))
        ),
        # Time com superioridade numérica entre a bola e o gol quando é o time adversário atacando
        'opponent_team_attacking_dir_numerical_superiority_team': (
                F.when(F.col('opponentTeamAttackingDirection') == 'Left', F.col('numerical_superiority_team_left'))
                .when(F.col('opponentTeamAttackingDirection') == 'Right', F.col('numerical_superiority_team_right'))
        ),
    }).drop('homePlayers_between_ball_left_goal', 'awayPlayers_between_ball_left_goal', 'homePlayers_between_ball_right_goal', 'awayPlayers_between_ball_right_goal', 'players_between_ball_left_goal', 'players_between_ball_right_goal', 'numerical_superiority_team_left', 'numerical_superiority_team_right')
)

df_events_qtd_players.show()

+------+-------------+---------+--------------------+---------+--------------------+------+-----------------+-----------------------+--------------+--------------------+-------------+-----------------+-----------+---------------+--------------------+--------------------+--------------------+----------+----------+---------------+--------------+----------------+--------------------------+-----------------+----------------+-------------+------------+--------------------------+------------------------------+-------------+--------------------+-------------+------------------+-------------------+----------------+-------------------+-------------------------------------+-----------------------------------------+-------------------------------------------------+-----------------------------------------------------+--------------------------------------------------+------------------------------------------------------+
|gameId|competitionId|   season|             eventId|eventType|eventTypeDes

In [ ]:
# agregação das colunas de ameaça para criar o threat score